In [1]:
import os
from typing import Tuple, Dict, Any

import numpy as np
import pandas as pd

from stable_baselines3 import PPO, SAC

from sentiment_enviroment import SentimentEnv
from technical_enviroment import TechnicalEnv
from super_agent_envoriment import SuperAgentEnv
from meta_agent_enviroment import MetaAgentEnv
from custom_function import split_data_chronologically, add_regime_indicator
from agent_wrapper import AgentWrapper, EarlyStoppingCallback, evaluate_agent
import warnings

warnings.filterwarnings('ignore')


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Training pipeline for the SuperAgent and MetaAgent.

This module demonstrates how to load pre‑trained base agents (sentiment and
technical), wrap them with the provided AgentWrapper class, define the
SuperAgent and MetaAgent environments, and train higher‑level agents on
chronologically split data.  The pipeline follows the same structure as the
base agent training utilities found in ``agent_wrapper.py`` and
``wrapper_agent.py`` but extends it to the hierarchical reinforcement
learning setup.

**Usage**

1. Prepare your datasets for monthly prices, technical indicators, and
   sentiment features.  These should be pandas DataFrames indexed by
   datetime with matching lengths and column ordering across the datasets.
2. Train your base agents (sentiment and technical) using the
   ``train_and_evaluate_with_split`` function from ``agent_wrapper.py`` or
   ``wrapper_agent.py``.  Save the best performing models to disk.
3. Adjust the file paths in the ``load_pretrained_agents`` function to point
   at your saved models.  You may also modify the hyperparameters
   ``super_ppo_params`` and ``meta_ppo_params`` to suit your needs.
4. Run this script.  It will split your data into train/validation/test
   segments, instantiate the environments, train a SuperAgent to blend the
   recommendations of the base agents, and then train a MetaAgent to make
   the final portfolio decisions.  After training, the script prints
   evaluation metrics for each layer of the hierarchy.

Because this script requires data and pre‑trained models that are not
included in this repository, it is intended as a template rather than a
stand‑alone executable.  Replace the placeholder file paths and tune
hyperparameters as appropriate for your project.


In [2]:
def load_pretrained_agents(
    sentiment_model_path: str,
    technical_model_path: str,
    sentiment_train_env: SentimentEnv,
    technical_train_env: TechnicalEnv
) -> Tuple[AgentWrapper, AgentWrapper]:
    """
    Load pre‑trained sentiment and technical agents from disk and wrap them
    with the AgentWrapper class.  If the specified files do not exist,
    this function will raise a FileNotFoundError.

    Args:
        sentiment_model_path: Path to the saved sentiment model (.zip)
        technical_model_path: Path to the saved technical model (.zip)
        sentiment_train_env: A training environment instance for the
            sentiment agent.  The loaded model will be bound to this
            environment.
        technical_train_env: A training environment instance for the
            technical agent.  The loaded model will be bound to this
            environment.

    Returns:
        Tuple of (sentiment_agent_wrapper, technical_agent_wrapper)
    """
    if not os.path.exists(sentiment_model_path):
        raise FileNotFoundError(f"Sentiment model not found at {sentiment_model_path}")
    if not os.path.exists(technical_model_path):
        raise FileNotFoundError(f"Technical model not found at {technical_model_path}")

    # Determine algorithm type based on file naming convention
    # Stable‑baselines3 stores algorithm info in the zip; we need to know
    # which class to instantiate for loading.  Infer from file name.
    if 'ppo' in sentiment_model_path.lower():
        sentiment_model = PPO.load(sentiment_model_path, env=sentiment_train_env)
        sentiment_algo = 'PPO'
    else:
        sentiment_model = SAC.load(sentiment_model_path, env=sentiment_train_env)
        sentiment_algo = 'SAC'

    if 'ppo' in technical_model_path.lower():
        technical_model = PPO.load(technical_model_path, env=technical_train_env)
        technical_algo = 'PPO'
    else:
        technical_model = SAC.load(technical_model_path, env=technical_train_env)
        technical_algo = 'SAC'

    sentiment_agent = AgentWrapper(sentiment_model, sentiment_train_env, sentiment_algo, 'sentiment')
    technical_agent = AgentWrapper(technical_model, technical_train_env, technical_algo, 'technical')

    # Immediately reset to initialize internal weight vectors
    sentiment_agent.reset()
    technical_agent.reset()

    return sentiment_agent, technical_agent

In [3]:

def build_super_agent_envs(
    train_prices: pd.DataFrame,
    val_prices: pd.DataFrame,
    test_prices: pd.DataFrame,
    sentiment_train_env: SentimentEnv,
    sentiment_val_env: SentimentEnv,
    sentiment_test_env: SentimentEnv,
    technical_train_env: TechnicalEnv,
    technical_val_env: TechnicalEnv,
    technical_test_env: TechnicalEnv,
    sentiment_agent: AgentWrapper,
    technical_agent: AgentWrapper,
    regime_indicators: Dict[str, pd.DataFrame] = None,
    initial_capital: float = 100_000
) -> Tuple[SuperAgentEnv, SuperAgentEnv, SuperAgentEnv]:
    """
    Instantiate SuperAgentEnv environments for the train, validation, and
    test splits.  Each SuperAgentEnv observes the latest weights from the
    base sentiment and technical agents and learns to blend them to
    maximize the composite reward.

    Args:
        train_prices, val_prices, test_prices: Price data for each split.
        sentiment_train_env, sentiment_val_env, sentiment_test_env: The
            corresponding base sentiment environments.  These are used to
            look up sentiment features and maintain internal indices for
            their agents.
        technical_train_env, technical_val_env, technical_test_env: The
            corresponding base technical environments.
        sentiment_agent, technical_agent: The pre‑trained base agents.  The
            same instances are shared across splits to preserve weights but
            their environments are updated by resetting them before use.
        regime_indicators: Optional dictionary mapping split names to
            DataFrames of market regime indicators aligned with the price
            index.  Keys can be 'train', 'val', or 'test'.
        initial_capital: Starting capital for the super agent.

    Returns:
        Tuple of (super_train_env, super_val_env, super_test_env)
    """
    # Prepare regime indicators for each split if provided
    regime_train = regime_indicators.get('train') if regime_indicators else None
    regime_val = regime_indicators.get('val') if regime_indicators else None
    regime_test = regime_indicators.get('test') if regime_indicators else None

    # Create environments
    super_train_env = SuperAgentEnv(
        price_data=train_prices,
        sentiment_agent=sentiment_agent,
        technical_agent=technical_agent,
        regime_indicators=regime_train,
        initial_capital=initial_capital,
    )
    super_val_env = SuperAgentEnv(
        price_data=val_prices,
        sentiment_agent=sentiment_agent,
        technical_agent=technical_agent,
        regime_indicators=regime_val,
        initial_capital=initial_capital,
    )
    super_test_env = SuperAgentEnv(
        price_data=test_prices,
        sentiment_agent=sentiment_agent,
        technical_agent=technical_agent,
        regime_indicators=regime_test,
        initial_capital=initial_capital,
    )
    return super_train_env, super_val_env, super_test_env


def build_meta_agent_envs(
    train_prices: pd.DataFrame,
    val_prices: pd.DataFrame,
    test_prices: pd.DataFrame,
    train_technical: pd.DataFrame,
    val_technical: pd.DataFrame,
    test_technical: pd.DataFrame,
    train_sentiment: pd.DataFrame,
    val_sentiment: pd.DataFrame,
    test_sentiment: pd.DataFrame,
    sentiment_agent: AgentWrapper,
    technical_agent: AgentWrapper,
    super_agent_model: Any,
    super_train_env: SuperAgentEnv,
    super_val_env: SuperAgentEnv,
    super_test_env: SuperAgentEnv,
    regime_indicators: Dict[str, pd.DataFrame] = None,
    initial_capital: float = 100_000
) -> Tuple[MetaAgentEnv, MetaAgentEnv, MetaAgentEnv]:
    """
    Instantiate MetaAgentEnv environments for the train, validation, and
    test splits.  The meta agent observes the original technical and
    sentiment features, the current weights from all subordinate agents
    (sentiment, technical, and super), and optional market regime
    indicators.  It learns to make the final portfolio allocation.

    Args:
        train_prices, val_prices, test_prices: Price data for each split.
        train_technical, val_technical, test_technical: Technical feature
            matrices for each split.
        train_sentiment, val_sentiment, test_sentiment: Sentiment feature
            matrices for each split.
        sentiment_agent, technical_agent: Pre‑trained base agents.
        super_agent_model: The trained RL model used by the super agent.
        super_train_env, super_val_env, super_test_env: The
            environments corresponding to each split for the super agent.
        regime_indicators: Optional dictionary of regime indicators for
            each split.
        initial_capital: Starting capital for the meta agent.

    Returns:
        Tuple of (meta_train_env, meta_val_env, meta_test_env)
    """
    # Wrap the super agent model for use in the MetaAgentEnv.  Since the
    # super agent is trained by stable‑baselines3, its predict method
    # already returns valid actions.  We need to wrap it with the
    # AgentWrapper so that it exposes a ``weights`` attribute and can
    # maintain state across steps.  The ``algorithm_name`` argument is
    # inferred from the class type.
    if isinstance(super_agent_model, PPO):
        super_algo = 'PPO'
    else:
        super_algo = 'SAC'
    super_agent_wrapper_train = AgentWrapper(super_agent_model, super_train_env, super_algo, 'super')
    super_agent_wrapper_val = AgentWrapper(super_agent_model, super_val_env, super_algo, 'super')
    super_agent_wrapper_test = AgentWrapper(super_agent_model, super_test_env, super_algo, 'super')

    # Reset all wrappers to initialize weights
    super_agent_wrapper_train.reset()
    super_agent_wrapper_val.reset()
    super_agent_wrapper_test.reset()

    # Extract regime indicators per split
    regime_train = regime_indicators.get('train') if regime_indicators else None
    regime_val = regime_indicators.get('val') if regime_indicators else None
    regime_test = regime_indicators.get('test') if regime_indicators else None

    meta_train_env = MetaAgentEnv(
        price_data=train_prices,
        features=train_technical,
        sentiment_features=train_sentiment,
        sentiment_agent=sentiment_agent,
        technical_agent=technical_agent,
        super_agent=super_agent_wrapper_train,
        regime_indicators=regime_train,
        initial_capital=initial_capital,
    )
    meta_val_env = MetaAgentEnv(
        price_data=val_prices,
        features=val_technical,
        sentiment_features=val_sentiment,
        sentiment_agent=sentiment_agent,
        technical_agent=technical_agent,
        super_agent=super_agent_wrapper_val,
        regime_indicators=regime_val,
        initial_capital=initial_capital,
    )
    meta_test_env = MetaAgentEnv(
        price_data=test_prices,
        features=test_technical,
        sentiment_features=test_sentiment,
        sentiment_agent=sentiment_agent,
        technical_agent=technical_agent,
        super_agent=super_agent_wrapper_test,
        regime_indicators=regime_test,
        initial_capital=initial_capital,
    )
    return meta_train_env, meta_val_env, meta_test_env


def train_super_agent(
    super_train_env: SuperAgentEnv,
    super_val_env: SuperAgentEnv,
    timesteps: int = 50_000,
    algorithm: str = 'ppo',
    ppo_params: Dict[str, Any] = None,
    sac_params: Dict[str, Any] = None,
    early_stopping: bool = True,
    early_stopping_params: Dict[str, Any] = None,
    save_path: str = './models/super'
) -> Tuple[Any, EarlyStoppingCallback]:
    """
    Train a super agent on the provided training environment.  The
    algorithm can be either PPO or SAC, and hyperparameters may be
    supplied via ``ppo_params`` or ``sac_params``.  Optionally performs
    early stopping based on validation Sharpe ratio.

    Args:
        super_train_env: Training environment for the super agent.
        super_val_env: Validation environment for early stopping.
        timesteps: Total number of training steps.
        algorithm: Either 'ppo' or 'sac'.
        ppo_params: Hyperparameters for PPO; if None, defaults will be used.
        sac_params: Hyperparameters for SAC; if None, defaults will be used.
        early_stopping: If True, enable early stopping on validation Sharpe.
        early_stopping_params: Dictionary of parameters for the
            EarlyStoppingCallback (e.g. eval_freq, patience, min_delta).
        save_path: Directory to save model checkpoints and plots.

    Returns:
        Tuple of (trained_model, callback)
    """
    os.makedirs(save_path, exist_ok=True)
    algorithm = algorithm.lower()
    if algorithm not in ('ppo', 'sac'):
        raise ValueError("algorithm must be 'ppo' or 'sac'")

    # Instantiate model
    if algorithm == 'ppo':
        params = ppo_params or {}
        model = PPO('MlpPolicy', super_train_env, **params)
    else:
        params = sac_params or {}
        model = SAC('MlpPolicy', super_train_env, **params)

    # Configure early stopping callback
    if early_stopping:
        es_params = early_stopping_params or {}
        callback = EarlyStoppingCallback(
            val_env=super_val_env,
            eval_freq=es_params.get('eval_freq', 2_000),
            patience=es_params.get('patience', 5),
            min_delta=es_params.get('min_delta', 0.01),
            save_path=os.path.join(save_path, 'checkpoint'),
            verbose=1,
        )
    else:
        callback = None

    # Train the model
    model.learn(total_timesteps=timesteps, callback=callback)

    # Load the best model if early stopping saved one
    if early_stopping and callback and callback.best_model_path:
        if algorithm == 'ppo':
            model = PPO.load(callback.best_model_path, env=super_train_env)
        else:
            model = SAC.load(callback.best_model_path, env=super_train_env)

    return model, callback


def train_meta_agent(
    meta_train_env: MetaAgentEnv,
    meta_val_env: MetaAgentEnv,
    timesteps: int = 50_000,
    algorithm: str = 'ppo',
    ppo_params: Dict[str, Any] = None,
    sac_params: Dict[str, Any] = None,
    early_stopping: bool = True,
    early_stopping_params: Dict[str, Any] = None,
    save_path: str = './models/meta'
) -> Tuple[Any, EarlyStoppingCallback]:
    """
    Train a meta agent on the provided training environment.  The
    algorithm can be either PPO or SAC, and hyperparameters may be
    supplied via ``ppo_params`` or ``sac_params``.  Optionally performs
    early stopping based on validation Sharpe ratio.

    Args:
        meta_train_env: Training environment for the meta agent.
        meta_val_env: Validation environment for early stopping.
        timesteps: Total number of training steps.
        algorithm: Either 'ppo' or 'sac'.
        ppo_params: Hyperparameters for PPO; if None, defaults will be used.
        sac_params: Hyperparameters for SAC; if None, defaults will be used.
        early_stopping: If True, enable early stopping on validation Sharpe.
        early_stopping_params: Dictionary of parameters for the
            EarlyStoppingCallback.
        save_path: Directory to save model checkpoints and plots.

    Returns:
        Tuple of (trained_model, callback)
    """
    os.makedirs(save_path, exist_ok=True)
    algorithm = algorithm.lower()
    if algorithm not in ('ppo', 'sac'):
        raise ValueError("algorithm must be 'ppo' or 'sac'")

    # Instantiate model
    if algorithm == 'ppo':
        params = ppo_params or {}
        model = PPO('MlpPolicy', meta_train_env, **params)
    else:
        params = sac_params or {}
        model = SAC('MlpPolicy', meta_train_env, **params)

    # Configure early stopping callback
    if early_stopping:
        es_params = early_stopping_params or {}
        callback = EarlyStoppingCallback(
            val_env=meta_val_env,
            eval_freq=es_params.get('eval_freq', 2_000),
            patience=es_params.get('patience', 5),
            min_delta=es_params.get('min_delta', 0.01),
            save_path=os.path.join(save_path, 'checkpoint'),
            verbose=1,
        )
    else:
        callback = None

    # Train the model
    model.learn(total_timesteps=timesteps, callback=callback)

    # Load the best model if early stopping saved one
    if early_stopping and callback and callback.best_model_path:
        if algorithm == 'ppo':
            model = PPO.load(callback.best_model_path, env=meta_train_env)
        else:
            model = SAC.load(callback.best_model_path, env=meta_train_env)

    return model, callback



In [4]:
price_data = pd.read_csv('output_data/price_data.csv', index_col=0, parse_dates=True)
technical_features = pd.read_csv('output_data/technical_features.csv', index_col=0, parse_dates=True)
sentiment_features = pd.read_csv('output_data/sentiment_features.csv', index_col=0, parse_dates=True)
regime = pd.read_csv('output_data/regime_indicators.csv', index_col=0, parse_dates=True)


splits = split_data_chronologically(price_data, technical_features, sentiment_features)
(train_prices, train_technical, train_sentiment) = splits['train']
(val_prices, val_technical, val_sentiment) = splits['val']
(test_prices, test_technical, test_sentiment) = splits['test']

# Build regime splits if applicable
regime_splits = None
if regime is not None:
    regime_train = regime.iloc[: len(train_prices)]
    regime_val = regime.iloc[len(train_prices): len(train_prices) + len(val_prices)]
    regime_test = regime.iloc[len(train_prices) + len(val_prices):]
    regime_splits = {
        'train': regime_train,
        'val': regime_val,
        'test': regime_test,
    }

# ------------------------------------------------------------------
# Step 3: Instantiate base environments for splitting
# ------------------------------------------------------------------
sentiment_train_env = SentimentEnv(train_prices, train_sentiment)
sentiment_val_env = SentimentEnv(val_prices, val_sentiment)
sentiment_test_env = SentimentEnv(test_prices, test_sentiment)

technical_train_env = TechnicalEnv(train_prices, train_technical)
technical_val_env = TechnicalEnv(val_prices, val_technical)
technical_test_env = TechnicalEnv(test_prices, test_technical)

# ------------------------------------------------------------------
# Step 4: Load pre‑trained base agents

sentiment_model_path = 'models/Sentiment_PPO_best.zip'
technical_model_path = 'models/Technical_PPO_best.zip'
sentiment_agent, technical_agent = load_pretrained_agents(
    sentiment_model_path,
    technical_model_path,
    sentiment_train_env,
    technical_train_env
)

super_train_env, super_val_env, super_test_env = build_super_agent_envs(
    train_prices,
    val_prices,
    test_prices,
    sentiment_train_env,
    sentiment_val_env,
    sentiment_test_env,
    technical_train_env,
    technical_val_env,
    technical_test_env,
    sentiment_agent,
    technical_agent,
    regime_indicators=regime_splits,
)

# Define hyperparameters for super agent training
super_ppo_params = {
    'learning_rate': 0.0005,
    'n_steps': 2048,
    'batch_size': 128,
    'n_epochs': 3,
    'gamma': 0.95,
    'gae_lambda': 0.9,
    'ent_coef': 0.1,
    'vf_coef': 0.5,
    'max_grad_norm': 0.5,
    'clip_range': 0.2,
    'verbose': 0,
}
super_model, super_callback = train_super_agent(
    super_train_env,
    super_val_env,
    timesteps=100_000,
    algorithm='ppo',
    ppo_params=super_ppo_params,
    early_stopping=True,
    early_stopping_params={'eval_freq': 1_500, 'patience': 5, 'min_delta': 0.01},
    save_path='./models/super',
)

# Evaluate the super agent on the test set
super_test_metrics = evaluate_agent(super_model, super_test_env, phase='Super Test')



DATA SPLITTING

Total data: 129 months
Date range: 2015-02-28 00:00:00 to 2025-10-31 00:00:00

DATA SPLIT SUMMARY:
Train:  77 months (60%) | 2015-02 to 2021-06
Val:    25 months (20%) | 2021-07 to 2023-07
Test:   27 months (20%) | 2023-08 to 2025-10

No data leakage - splits are properly separated
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Validation (Callback) Results:
  Sharpe Ratio:    0.107
  Total Return:   -6.54%
  Max Drawdown:   47.49%
  Win Rate:       50.00%
Step 1500: Val Sharpe = 0.107
  New best! Saved model (Sharpe: 0.107)

Validation (Callback) Results:
  Sharpe Ratio:   -0.182
  Total Return:  -19.21%
  Max Drawdown:   45.96%
  Win Rate:       50.00%
Step 3000: Val Sharpe = -0.182
  No improvement (patience: 1/5)

Validation (Callback) Results:
  Sharpe Ratio:    0.071
  Total Return:   -5.66%
  Max Drawdown:   47.95%
  Win Rate:       54.17%
Step 4500: Val S

In [5]:

print("Recreating super agent environments with updated code...")
super_train_env = SuperAgentEnv(train_prices, sentiment_agent, technical_agent, regime_indicators=regime_train)
super_val_env = SuperAgentEnv(val_prices, sentiment_agent, technical_agent, regime_indicators=regime_val)
super_test_env = SuperAgentEnv(test_prices, sentiment_agent, technical_agent, regime_indicators=regime_test)

# Verify shapes
print(f"Super train obs space: {super_train_env.observation_space.shape}")
print(f"Super val obs space: {super_val_env.observation_space.shape}")

# Build meta agent environments
print("\nBuilding meta agent environments...")
meta_train_env, meta_val_env, meta_test_env = build_meta_agent_envs(
    train_prices,
    val_prices,
    test_prices,
    train_technical,
    val_technical,
    test_technical,
    train_sentiment,
    val_sentiment,
    test_sentiment,
    sentiment_agent,
    technical_agent,
    super_model,
    super_train_env,
    super_val_env,
    super_test_env,
    regime_indicators=regime_splits,
)


# Define hyperparameters for meta agent training
meta_ppo_params = {
    'learning_rate': 0.0003,
    'n_steps': 2048,
    'batch_size': 128,
    'n_epochs': 3,
    'gamma': 0.95,
    'gae_lambda': 0.9,
    'ent_coef': 0.1,
    'vf_coef': 0.5,
    'max_grad_norm': 0.5,
    'clip_range': 0.2,
    'verbose': 0,
}

print("\nTraining meta agent...")
meta_model, meta_callback = train_meta_agent(
    meta_train_env,
    meta_val_env,
    timesteps=100_000,
    algorithm='ppo',
    ppo_params=meta_ppo_params,
    early_stopping=True,
    early_stopping_params={'eval_freq': 1_500, 'patience': 5, 'min_delta': 0.01},
    save_path='./models/meta',
)

# Evaluate the meta agent on the test set
meta_test_metrics = evaluate_agent(meta_model, meta_test_env, phase='Meta Test')


Recreating super agent environments with updated code...
Super train obs space: (30,)
Super val obs space: (30,)

Building meta agent environments...

Training meta agent...

Validation (Callback) Results:
  Sharpe Ratio:   -0.309
  Total Return:  -46.57%
  Max Drawdown:   63.50%
  Win Rate:       41.67%
Step 1500: Val Sharpe = -0.309
  New best! Saved model (Sharpe: -0.309)

Validation (Callback) Results:
  Sharpe Ratio:   -0.168
  Total Return:  -23.98%
  Max Drawdown:   40.59%
  Win Rate:       41.67%
Step 3000: Val Sharpe = -0.168
  New best! Saved model (Sharpe: -0.168)

Validation (Callback) Results:
  Sharpe Ratio:    0.046
  Total Return:  -10.66%
  Max Drawdown:   42.88%
  Win Rate:       58.33%
Step 4500: Val Sharpe = 0.046
  New best! Saved model (Sharpe: 0.046)

Validation (Callback) Results:
  Sharpe Ratio:    0.048
  Total Return:  -10.49%
  Max Drawdown:   42.88%
  Win Rate:       58.33%
Step 6000: Val Sharpe = 0.048
  No improvement (patience: 1/5)

Validation (Callback

In [9]:
print("\n" + "="*70)
print("FINAL RESULTS COMPARISON")
print("="*70)
print("\nSuper Agent Test Metrics:")
for k, v in super_test_metrics.items():
    print(f"  {k:15s}: {v:>8.4f}")
print("\nMeta Agent Test Metrics:")
for k, v in meta_test_metrics.items():
    print(f"  {k:15s}: {v:>8.4f}")


FINAL RESULTS COMPARISON

Super Agent Test Metrics:
  total_return   :   1.3290
  sharpe_ratio   :   1.5232
  max_drawdown   :   0.1363
  volatility     :   0.2851
  final_value    : 232902.8015
  win_rate       :   0.6538

Meta Agent Test Metrics:
  total_return   :   1.1798
  sharpe_ratio   :   1.3256
  max_drawdown   :   0.2017
  volatility     :   0.3080
  final_value    : 217982.8531
  win_rate       :   0.5000


In [8]:


# Performance improvement analysis
print("\n" + "="*70)
print("META AGENT IMPROVEMENT OVER SUPER AGENT")
print("="*70)
sharpe_improvement = ((meta_test_metrics['sharpe_ratio'] - super_test_metrics['sharpe_ratio']) 
                      / super_test_metrics['sharpe_ratio'] * 100)
return_improvement = ((meta_test_metrics['total_return'] - super_test_metrics['total_return']) 
                      / super_test_metrics['total_return'] * 100)
print(f"Sharpe Ratio Improvement: {sharpe_improvement:+.2f}%")
print(f"Total Return Improvement: {return_improvement:+.2f}%")


META AGENT IMPROVEMENT OVER SUPER AGENT
Sharpe Ratio Improvement: -12.97%
Total Return Improvement: -11.23%
